# Huấn luyện mô hình BARTPho trên bộ dữ liệu VietNews (30k mẫu, 3 Epochs)

File Colab Notebook này được thiết kế để fine-tune mô hình `vinai/bartpho-syllable` trên bộ dữ liệu tiếng Việt `nam194/vietnews` với **30,000 mẫu huấn luyện** trong **3 epochs**.

### Các bước chuẩn bị:
1. Chọn môi trường chạy (Runtime): **GPU (L4, A100 hoặc ít nhất là T4)**.
2. Mount Google Drive để tự động lưu checkpoints phòng trường hợp Colab bị ngắt kết nối.
3. Chạy từng ô lệnh bên dưới theo thứ tự.

## Bước 1: Cài đặt thư viện và Thiết lập cấu hình

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q transformers==4.52.4 tokenizers==0.21.1 sentencepiece==0.2.0 datasets evaluate rouge-score bert-score accelerate safetensors

import inspect
import os
import random
import shutil
import warnings
from pathlib import Path

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import load_dataset
from IPython.display import display
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Running on device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Cấu hình huấn luyện (BARTPho, VietNews, 30,000 mẫu, 3 Epochs)
CFG = {
    "model_name": "vinai/bartpho-syllable",
    "checkpoint_path": "/content/drive/MyDrive/checkpoint-4000",  # Đường dẫn lối tắt của checkpoint-4000 trong My Drive
    "dataset_name": "nam194/vietnews",
    "source_col": "article",
    "target_col": "abstract",
    "prefix": "",                      # BARTPho không cần prefix giống họ model T5
    "max_src_len": 512,
    "max_tgt_len": 128,
    "epochs": 3,
    "batch_size": 4,
    "eval_batch_size": 4,
    "grad_acc_steps": 4,
    "lr": 3e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.03,
    "seed": 42,
    "train_sample_size": 30000,
    "val_sample_size": 5000,
    "test_sample_size": 1000,
    "output_dir": "./bartpho-colab-checkpoints",
    "final_dir": "./bartpho-colab-finetuned",
    "zip_name": "bartpho-colab-finetuned.zip",
    "save_to_drive": True,
    "drive_dir": "/content/drive/MyDrive/bartpho-training",
    "eval_strategy": "steps",
    "eval_steps": 1000,
    "save_strategy": "steps",
    "save_steps": 1000,
    "no_resume": False,
    "use_fast": False,                  # use_fast=False là bắt buộc đối với BARTPho
}

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])
print("Cấu hình thành công:", CFG)


## Bước 2: Tải và Rút trích dữ liệu VietNews

In [ ]:
print("[Data] Đang tải dataset từ HuggingFace...")
dataset = load_dataset(CFG["dataset_name"])
print(dataset)

def pick_split(name: str, fallback: str):
    if name in dataset:
        return dataset[name]
    return dataset[fallback]

train_full = pick_split("train", list(dataset.keys())[0])
val_full = pick_split("validation", "train")
test_full = pick_split("test", "validation" if "validation" in dataset else "train")

def safe_select(ds, n: int):
    n = min(n, len(ds))
    return ds.shuffle(seed=CFG["seed"]).select(range(n))

train_data = safe_select(train_full, CFG["train_sample_size"])
val_data = safe_select(val_full, CFG["val_sample_size"])
test_data = safe_select(test_full, CFG["test_sample_size"])

print("\n=== Kích thước tập dữ liệu huấn luyện ===")
print(f"Tập Train (Huấn luyện) : {len(train_data):,} mẫu")
print(f"Tập Val (Kiểm định)     : {len(val_data):,} mẫu")
print(f"Tập Test (Kiểm thử)    : {len(test_data):,} mẫu")
print("\nCác trường dữ liệu:", train_data.column_names)

sample_df = pd.DataFrame(train_data[:3])
display(sample_df[[CFG["source_col"], CFG["target_col"]]])

## Bước 3: Phân tích phân bổ độ dài Token

In [ ]:
# Kích hoạt Google Drive sớm để đọc checkpoint từ lối tắt
try:
    from google.colab import drive
    import os
    print('[Drive] Đang kết nối tới Google Drive...')
    drive.mount('/content/drive')
except Exception as exc:
    print(f'[Drive] Không thể kết nối Google Drive: {exc}')

# Kiểm tra đường dẫn checkpoint
model_load_path = CFG.get("checkpoint_path", CFG["model_name"])
if model_load_path.startswith("/content/drive") and not os.path.exists(model_load_path):
    print(f'[Warn] Không tìm thấy checkpoint tại {model_load_path}. Sẽ sử dụng model gốc mặc định: {CFG["model_name"]}')
    model_load_path = CFG["model_name"]
else:
    print(f'[Load] Sẽ tải mô hình và tokenizer từ: {model_load_path}')

tokenizer = AutoTokenizer.from_pretrained(model_load_path, use_fast=CFG.get("use_fast", False))


## Bước 4: Tiền xử lý và Tokenize dữ liệu

In [ ]:
def preprocess_function(examples):
    inputs = [
        CFG["prefix"] + str(doc).strip()
        for doc in examples[CFG["source_col"]]
    ]
    targets = [
        str(summary).strip()
        for summary in examples[CFG["target_col"]]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=CFG["max_src_len"],
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=targets,
        max_length=CFG["max_tgt_len"],
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

remove_cols = train_data.column_names
print("[Process] Đang tokenize tập dữ liệu...")
tokenized_train = train_data.map(
    preprocess_function,
    batched=True,
    batch_size=128,
    remove_columns=remove_cols,
    desc="Tokenizing train",
)
tokenized_val = val_data.map(
    preprocess_function,
    batched=True,
    batch_size=128,
    remove_columns=remove_cols,
    desc="Tokenizing validation",
)
tokenized_test = test_data.map(
    preprocess_function,
    batched=True,
    batch_size=128,
    remove_columns=remove_cols,
    desc="Tokenizing test",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=CFG["model_name"],
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)
print("Tiền xử lý hoàn tất.")

## Bước 5: Khởi tạo Mô hình và Tiến hành Huấn luyện

In [ ]:
# Load model từ checkpoint_path nếu hợp lệ (đã gán vào model_load_path ở bước Tokenizer)
model = AutoModelForSeq2SeqLM.from_pretrained(model_load_path)
model.config.use_cache = False
model.to(DEVICE)
print(f"Tổng số tham số mô hình: {sum(p.numel() for p in model.parameters()):,}")

# Tải công cụ tính ROUGE Score
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    predictions = np.where(predictions != -100, predictions, pad_token_id)
    labels = np.where(labels != -100, labels, pad_token_id)
    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False,
    )
    return {k: round(v * 100, 4) for k, v in result.items()}

# Kết nối Google Drive để lưu trữ checkpoint
training_output_dir = CFG["output_dir"]
if CFG.get("save_to_drive", False):
    try:
        from google.colab import drive
        print("[Drive] Đang kết nối tới Google Drive...")
        drive.mount("/content/drive")
        drive_output_dir = Path(CFG["drive_dir"]) / "checkpoints"
        drive_output_dir.mkdir(parents=True, exist_ok=True)
        training_output_dir = str(drive_output_dir)
        print(f"[Drive] Checkpoints sẽ được lưu trực tiếp trên Google Drive: {training_output_dir}")
    except Exception as exc:
        print(f"[Drive] Không thể kết nối Google Drive, chuyển về lưu tại bộ nhớ tạm: {exc}")
        training_output_dir = CFG["output_dir"]

def build_training_args():
    # Kiểm tra hỗ trợ BF16 trên GPU hiện tại
    bf16_supported = False
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        if major >= 8:
            bf16_supported = True

    # BARTPho chạy ổn định bằng FP16 mixed precision (không lo bị NaN loss giống các mô hình họ T5)
    if bf16_supported:
        use_fp16 = False
        use_bf16 = True
        print("[Optimizer] Phát hiện GPU hỗ trợ BF16. Sử dụng BF16 + AdamW.")
    else:
        use_fp16 = True
        use_bf16 = False
        print("[Optimizer] GPU không hỗ trợ BF16 (như T4). Sử dụng FP16 mixed precision + AdamW (BARTPho chạy cực kỳ ổn định với FP16).")

    kwargs = {
        "output_dir": training_output_dir,
        "learning_rate": CFG["lr"],
        "per_device_train_batch_size": CFG["batch_size"],
        "per_device_eval_batch_size": CFG["eval_batch_size"],
        "gradient_accumulation_steps": CFG["grad_acc_steps"],
        "weight_decay": CFG["weight_decay"],
        "save_total_limit": 2,
        "num_train_epochs": CFG["epochs"],
        "predict_with_generate": True,
        "generation_max_length": CFG["max_tgt_len"],
        "logging_steps": 100,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "report_to": "none",
        "warmup_ratio": CFG["warmup_ratio"],
        "dataloader_num_workers": 0,
        "gradient_checkpointing": True,
        "fp16": use_fp16,
        "bf16": use_bf16,
        "optim": "adamw_torch",
    }

    # Điều chỉnh tên tham số evaluate theo phiên bản transformers
    signature = inspect.signature(Seq2SeqTrainingArguments.__init__)
    if "eval_strategy" in signature.parameters:
        kwargs["eval_strategy"] = CFG.get("eval_strategy", "steps")
    else:
        kwargs["evaluation_strategy"] = CFG.get("eval_strategy", "steps")

    kwargs["save_strategy"] = CFG.get("save_strategy", "steps")
    kwargs["save_steps"] = CFG.get("save_steps", 1000)
    kwargs["eval_steps"] = CFG.get("eval_steps", 1000)

    return Seq2SeqTrainingArguments(**kwargs)

training_args = build_training_args()

from transformers import EarlyStoppingCallback
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Tiếp tục huấn luyện từ checkpoint cũ nếu có
resume_from_checkpoint = None
if model_load_path.startswith("/content/drive") and os.path.exists(model_load_path):
    resume_from_checkpoint = model_load_path
    print(f"[Trainer] Phát hiện checkpoint từ Drive. Sẽ tiếp tục huấn luyện từ: {resume_from_checkpoint}")

if resume_from_checkpoint is None and not CFG.get("no_resume", False):
    checkpoint_dir = Path(training_output_dir)
    if checkpoint_dir.exists():
        checkpoints = [d for d in checkpoint_dir.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")]
        if checkpoints:
            resume_from_checkpoint = True
            print(f"[Trainer] Phát hiện checkpoint cũ tại {training_output_dir}. Sẽ tiếp tục huấn luyện từ checkpoint mới nhất...")
        else:
            print(f"[Trainer] Không tìm thấy checkpoint cũ trong {training_output_dir}. Huấn luyện từ đầu.")
    else:
        print(f"[Trainer] Chưa có thư mục checkpoints. Huấn luyện từ đầu.")

print("[Trainer] Bắt đầu huấn luyện...")
trainer.train(resume_from_checkpoint=resume_from_checkpoint)

print("[Trainer] Tiến hành đánh giá trên tập validation...")
metrics = trainer.evaluate()
print("Validation Metrics:", metrics)

# Lưu model và tokenizer cuối cùng
final_dir = Path(CFG["final_dir"])
if final_dir.exists():
    shutil.rmtree(final_dir)
final_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

# Lưu báo cáo huấn luyện
metadata = {
    "base_model": CFG["model_name"],
    "dataset": CFG["dataset_name"],
    "source_col": CFG["source_col"],
    "target_col": CFG["target_col"],
    "max_src_len": CFG["max_src_len"],
    "max_tgt_len": CFG["max_tgt_len"],
    "epochs": CFG["epochs"],
    "train_sample_size": len(train_data),
    "val_sample_size": len(val_data),
    "metrics": metrics,
}
pd.Series(metadata, dtype="object").to_json(final_dir / "training_report.json", force_ascii=False, indent=2)
print(f"Đã lưu thành công model + tokenizer tại: {final_dir}")


## Bước 6: Kiểm thử sinh mẫu tóm tắt và Đóng gói

In [ ]:
def generate_summary(text: str) -> str:
    prefixed = CFG["prefix"] + str(text).strip()
    inputs = tokenizer(
        prefixed,
        return_tensors="pt",
        max_length=CFG["max_src_len"],
        truncation=True,
    ).to(DEVICE)
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        min_new_tokens=20,
        num_beams=2,
        no_repeat_ngram_size=5,
        repetition_penalty=2.5,
        length_penalty=1.05,
        early_stopping=True,
        do_sample=False,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()

# Sinh thử mẫu
test_samples = test_data.select(range(min(3, len(test_data))))
results = []
model_key = CFG["model_name"].split('/')[-1] + " prediction"

for i, example in enumerate(test_samples):
    original = example[CFG["source_col"]]
    reference = example[CFG["target_col"]]
    prediction = generate_summary(original)
    results.append({
        "ID": i + 1,
        "Bản gốc": str(original)[:260] + "...",
        "Tóm tắt chuẩn": reference,
        "BARTPho Tóm tắt": prediction,
    })

bertscore = evaluate.load("bertscore")
preds = [r["BARTPho Tóm tắt"] for r in results]
refs = [r["Tóm tắt chuẩn"] for r in results]
b_score = bertscore.compute(predictions=preds, references=refs, lang="vi")

for i in range(len(results)):
    results[i]["BERTScore F1"] = round(float(b_score["f1"][i]), 4)

df_results = pd.DataFrame(results)
display(df_results.style.set_properties(**{"text-align": "left", "vertical-align": "top"}))

# Đóng gói mô hình thành file zip để tải về hoặc chuyển về repo dự án
zip_path = Path(CFG["zip_name"])
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(CFG["zip_name"].replace(".zip", ""), "zip", root_dir=".", base_dir=CFG["final_dir"].lstrip("./"))
print(f"Đã đóng gói thành công file zip tại: {zip_path.resolve()}")

if CFG["save_to_drive"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_dir = Path(CFG["drive_dir"])
        drive_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zip_path, drive_dir / zip_path.name)
        print(f"[Drive] Đã sao lưu bản copy lên Google Drive tại: {drive_dir / zip_path.name}")
    except Exception as exc:
        print(f"[Drive] Không thể copy sang Drive: {exc}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print(f"[Download] Không thể tải tự động từ trình duyệt (vui lòng tải thủ công trong mục Files của Colab): {exc}")
